# Kaggle Session Template

Purpose: `STEP-031` session skeleton for the canonical Kaggle training workflow.

Order of operations:
1. Start session
2. Clone or pull repo
3. Verify environment
4. Verify attached datasets
5. Resume checkpoints if needed
6. Verify `config.yaml`
7. Train or resume
8. Evaluate
9. Export artifacts
10. Upload outputs to artifact tier
11. Push metadata back to Git
12. Shutdown


In [ ]:
from pathlib import Path

PLAN = Path('/kaggle/working/thesis/17_Automation/kaggle_sync/sync_plan.example.yaml')
REPO = Path('/kaggle/working/thesis')
EXPERIMENT_DIR = REPO / '06_Experiments' / 'EXP0001'
PLAN


## 1. Clone Or Pull The Repository

Runs the idempotent bootstrap step, installs `kaggle-requirements.txt` only when needed, and records attached dataset status.


In [ ]:
!python -m kaggle_sync --plan {PLAN} bootstrap


## 2. Verify Environment

This is the smoke check referenced by `environment/versions.lock.md`.


In [ ]:
import sys, torch, fastai, fastcore, numpy

print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'avail', torch.cuda.is_available())
print('fastai', fastai.__version__, 'fastcore', fastcore.__version__)
print('numpy', numpy.__version__)
assert torch.cuda.is_available(), 'GPU not visible'
assert sys.version_info[:2] == (3, 12)
x = torch.zeros(2, device='cuda').sum()
print('ok', float(x))


## 3. Verify Inputs And Resume Checkpoints

Kaggle datasets should already be attached to the notebook session before running training.


In [ ]:
!python -m kaggle_sync --plan {PLAN} resume-checkpoints
!ls /kaggle/input


## 4. Verify Experiment Config

Update `EXP0001` to the active experiment folder before training.


In [ ]:
import yaml

config_path = EXPERIMENT_DIR / 'config.yaml'
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
config


## 5. Train, Evaluate, Export

Run the experiment-specific training notebook or script here. Keep notebook outputs minimal and save durable outputs under the active `EXPxxxx` folder.


In [ ]:
# Example placeholder only:
# !python train.py --config 06_Experiments/EXP0001/config.yaml
# !python evaluate.py --experiment EXP0001
# !python export_model.py --experiment EXP0001


## 6. Publish Artifact-Tier Outputs And Push Metadata

Large outputs stay in Kaggle Dataset storage. Only source-tier metadata returns to Git.


In [ ]:
!python -m kaggle_sync --plan {PLAN} publish-artifacts
!python -m kaggle_sync --plan {PLAN} push-metadata --commit-message "sync: update EXP0001 metadata from Kaggle"
